# Module 16 — DVC + MLflow (embeddings)

Versionner le **modèle d'embedding**, mesurer **Recall@k**, pouvoir **rollback**.

Tuto : [`docs/modules/16-dvc-mlflow.md`](../docs/modules/16-dvc-mlflow.md) · ADR [0010](../docs/adr/0010-versionner-embedding.md)

**Prérequis cas B** : OpenSearch + Qdrant peuplés.

## Cas A — La métrique, sans infra

In [1]:
from presslake.mlops.params import load_embed_params
from presslake.mlops.recall import recall_at_k_from_flags
from presslake.mlops.registry import rollback_instructions
from presslake.vector.config import embedding_model, embedding_vector_size

print(load_embed_params())
print(embedding_model(), embedding_vector_size())
assert recall_at_k_from_flags([True, True, False], k=4) == 2 / 3
print(rollback_instructions().splitlines()[0])


{'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'vector_size': 384, 'top_k': 4}
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 384
Rollback embedding (remettre le registre, puis ré-embed) :


## Cas B — Recall@k sur le corpus (Qdrant + OpenSearch)

In [2]:
from presslake.mlops.recall import evaluate_recall, format_report, write_metrics

try:
    report = evaluate_recall()
except Exception as exc:
    print("Infra absente ou index vide :", type(exc).__name__, exc)
else:
    print(format_report(report))
    print("metrics →", write_metrics(report))


/home/anthony-marais/Documents/data_project/src/presslake/vector/embed.py:13: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  return TextEmbedding(model_name=embedding_model())


Recall@4  3/3 = 1.000
modèle     sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
dim        384
eval       rag-v1
refuse OK  2/2 (retrieve vide)
params     params.yaml
metrics → /home/anthony-marais/Documents/data_project/metrics/embed-recall.json
